<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - Digital Forensics</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 1: Introduction</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - What is Digital Forensics?</h1>

Digital forensics is the recovery, investigation, examination, and analysis of data stored on digital devices.

A large part of digital forensics involves understanding how data is stored, modified, hidden, and transmitted, along with knowing which tools and techniques can be used to analyze it effectively.

In CTF competitions, forensics challenges are often designed to emphasize creative problem solving, familiarity with forensic tools, and the ability to recognize unusual patterns. While real-world forensic investigations are typically more structured and procedural, CTF challenges tend to focus on experimentation and thinking outside the box.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - Examples of Digital Forensics in the Wild</h2>

#### BTK Killer

Dennis Rader, also known as the BTK Killer, was an American serial killer responsible for at least ten murders in Kansas between 1974 and 1991.

Over the course of the investigation, Rader communicated with police and local news outlets through letters and digital media. At one point, he asked investigators whether information stored on a floppy disk could be traced back to him. Police publicly responded that it could not.

Believing the disk was safe to use, Rader mailed a 1.44 MB floppy disk to a local television station. Investigators performed forensic analysis on the disk and recovered metadata from a deleted document stored on it. The metadata contained references to a church and the name “Dennis,” which ultimately helped investigators identify Rader and led to his arrest in 2005.

[Dennis Rader - Wikipedia](https://en.wikipedia.org/wiki/Dennis_Rader)

#### Operation Firewall

Another major example of digital forensics in action was Operation Firewall, an international investigation targeting the cybercrime organization Shadowcrew. Investigators used digital forensic techniques and network analysis to trace stolen credit card transactions, online communications, and other digital evidence associated with the group. The investigation ultimately led to dozens of arrests and the dismantling of a large identity theft and fraud operation.

[Original Blog Post](https://subrosacyber.com/en/blog/digital-forensics-examples)

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - Getting Set Up</h2>

Most forensics challenges that you will encounter require tools that can be installed directly from your operating system's package manager.

On Debian-based distributions, the following command will install many of the tools commonly used in CTF forensics challenges:

In [ ]:
!sudo apt update && sudo apt install binwalk xxd bsdextrautils binutils file libimage-exiftool-perl

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - What are Files?</h1>

At their core, files are simply sequences of bytes stored on a computer.

Different file formats define different ways of interpreting those bytes. For example, the same bytes could represent an image, a text document, an executable program, or compressed data depending on how they are interpreted.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 - Metadata</h2>

Metadata is “data about data.” It describes properties of a file rather than the main contents of the file itself.

Some metadata is stored by the filesystem or operating system, while other metadata may be embedded directly inside the file format.

Examples of metadata include:
- File size
- File timestamps (created, modified, accessed)
- File permissions
- Author information
- Camera model
- GPS coordinates
- Document creator software
- Comments or descriptions

Some file formats store metadata in structures near the beginning of the file, often called the file header, though metadata may also appear elsewhere in the file depending on the format.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 - File Signatures</h2>

A `file signature` (sometimes called `magic bytes`) is a sequence of bytes near the beginning of a file that helps identify the file format.

Some common examples include:

| File Type | Signature (Hex) |
|---|---|
| PNG | `89 50 4E 47` |
| JPG | `FF D8 FF` |
| PDF | `25 50 44 46` |
| ZIP | `50 4B 03 04` |
| ELF | `7F 45 4C 46` |

For more examples, see [List of Common File Signatures](https://en.wikipedia.org/wiki/List_of_file_signatures)

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.3 - Examining File Metadata</h2>

One of the first steps in digital forensics is identifying what a file actually is. File extensions can be misleading or intentionally modified, so analysts often inspect the raw contents of files directly.

#### `file` 

The `file` command attempts to identify the type of a file by examining its contents and file signatures rather than relying solely on the file extension.

Example:

In [ ]:
!file example_files/file.pdf

#### `hexdump` 

The `hexdump` utility displays the raw bytes of a file in hexadecimal format. This is extremely useful in digital forensics because it allows us to inspect the actual contents of a file rather than relying on how the operating system interprets it.

Lets take a look at the same file from earlier with hexdump:

In [ ]:
!hexdump -C example_files/file.pdf | head

NOTE: Because the output of hexdump is very long, we pipe the output to the head command to only display the first 20 lines.

We can see that the first 4 bytes of the file match the PDF file signature in the above table, indicating that this file is to be interpreted as a PDF. This is how the `file` command works internally.

#### `objdump`

The `objdump` utility can be used to examine object files and executables containing machine code. It provides information about a program’s internal structure, including:
- section layouts
- symbol tables
- assembly instructions
- program headers
- security-related properties

This can be useful in reverse engineering and binary forensics when analyzing compiled programs.

We can use the `-x` option to view all file headers in a simple ELF binary:

In [ ]:
!objdump -x example_files/test.elf

#### `exiftool`

The `exiftool` utility is used to examine and manipulate metadata stored inside files. It supports many file formats including:
- images
- PDFs
- audio files
- videos
- office documents

`exiftool` can reveal metadata such as:
- author names
- timestamps
- GPS coordinates
- camera information
- software used to create the file
- comments and descriptions

This is especially useful in digital forensics and CTF challenges where important information may be hidden inside metadata fields.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Challenge 1</h1>

*See /chal1 for files*

What is the name of the park that this photo was taken in?

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">3 - Modifying Files</h1>

Some forensic and CTF challenges require manually modifying the bytes of a file. This can be useful for repairing corrupted file headers, changing metadata, recovering damaged files, or exploring how file formats are structured internally.

There are many tools available for editing files at the byte level. Both Windows and Linux offer a variety of hex editors that allow users to view and modify raw file contents directly.

For this notebook, we will use [HexEd.it](https://hexed.it), a free browser-based hex editor.

Lets practice by modifiying a string in a simple ELF binary with HexEd.it.

We can also use python to do this in a more programmatic and repeatable way:

In [ ]:
# old and new must be the same length as to not corrupt the rest of the file
old = b"Hello world!!"
new = b"Hello RPISEC!" 

with open("example_files/test.elf", "rb") as f:
    data = f.read()

offset = data.find(old)
if offset == -1:
    raise ValueError("Original string not found")

patched = data[:offset] + new + data[offset + len(new):]

with open("example_files/patched.elf", "wb") as f:
    f.write(patched)
    print( "Patch Successful")

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Challenge 2</h1>

*See /chal2 for files*

Something happened to this file. See if you can do to recover its original state.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Bonus File Format - PCAP</h1>

Packet Capture (PCAP) files are one of the most common file formats encountered in network forensics challenges. A PCAP file contains a recording of network traffic, allowing investigators to examine packets that were sent and received on a network.

PCAP files can contain a wide variety of information, including:
- Web traffic (HTTP/HTTPS)
- DNS requests
- File transfers

**Wireshark** is the most popular graphical tool for analyzing packet captures.

We can analyze a simple pcap file with some HTTP traffic by opening it in wireshark.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Challenge 3</h1>

*See /chal3 for files*

We picked up some really weird network traffic between two parties. It appears that they are using some unknown protocol to communicate. Try to use what you have learned to reverse their protocol and recover any messages they have sent.